# Notebook 04: HuBERT-ECG Fine-tuning with PEFT

This notebook fine-tunes **HuBERT-ECG** (pretrained on 9.1 M ECGs from PhysioNet and other sources) on the PTB-XL 5-class diagnostic classification task.

Three parameter-efficient strategies are compared:

| Experiment | Strategy | What is updated |
|---|---|---|
| 1 | Selective unfreezing (8 blocks) | Last 8 of 12 transformer blocks + classifier head |
| 2 | LoRA r=8 | Low-rank adapters in attention Q/V projections (~0.8% of params) |
| 3 | DoRA r=8 | Weight-decomposed LoRA — magnitude + direction separated (~0.9%) |

Trained models are saved under `results/<experiment_name>/`. Results are loaded and visualised in `04c_comparison.ipynb`.

In [1]:
import sys, os, warnings
sys.path.append('../')
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

import torch
from src.utils.config               import CFG
from src.preprocessing.label_utils  import load_all_labels
from src.preprocessing.dataset_full import ECGDatasetFull
from src.models.hubert_ecg_finetune import HuBERTECGClassifier, HuBERTECGPEFT
from src.training.train_peft        import run_experiment

DATA_PATH = CFG['data']['path']
device    = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'GPU:   {torch.cuda.get_device_name(0)}')
print(f'VRAM:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Results dir: {CFG["paths"]["results"]}')

GPU:   NVIDIA GeForce RTX 4060 Laptop GPU
VRAM:  8.6 GB
Results dir: D:\GitHub\biosignal-xai\results/


## Data

PTB-XL contains 21,799 12-lead clinical ECGs labelled with up to five diagnostic superclasses: `NORM`, `MI`, `STTC`, `CD`, `HYP`. Records are split by stratified fold:
- Folds 1–8 → **train**
- Fold 9 → **validation** (used for early stopping and checkpoint selection)
- Fold 10 → **held-out test** (never touched during training)

HuBERT-ECG expects full 10-second recordings at 100 Hz, so `ECGDatasetFull` is used (output shape `(12, 1000)`). The model internally reshapes each record to process all 12 leads independently through the backbone.

In [2]:
Y = load_all_labels(
    DATA_PATH + 'ptbxl_database.csv',
    DATA_PATH + 'scp_statements.csv',
)
train_df = Y[Y.strat_fold <  9]
val_df   = Y[Y.strat_fold == 9]

train_ds = ECGDatasetFull(train_df, DATA_PATH)
val_ds   = ECGDatasetFull(val_df,   DATA_PATH)

print(f'Train: {len(train_df):,} records -> {len(train_ds):,} samples')
print(f'Val:   {len(val_df):,} records  -> {len(val_ds):,} samples')

Records with valid labels: 21388
Class distribution:
  NORM: 9514 (44.5%)
  MI: 5469 (25.6%)
  STTC: 5235 (24.5%)
  CD: 4898 (22.9%)
  HYP: 2649 (12.4%)
Train: 17,084 records -> 17,084 samples
Val:   2,146 records  -> 2,146 samples


## Experiment 1: Selective Unfreezing (8 Blocks)

HuBERT-ECG has 12 stacked transformer encoder blocks preceded by a convolutional feature extractor. Unfreezing the last 8 blocks (layers 4–11) exposes ~57 M of ~93 M total parameters. The convolutional feature extractor and the first 4 transformer blocks remain frozen, preserving low-level acoustic representations that transfer well across ECG tasks.

A warmup LR schedule is applied: LR ramps from half the target rate to the full rate over the first epoch, then cosine-decays. This avoids destructive updates to pretrained weights in the first steps.

In [3]:
model_8b = HuBERTECGClassifier(
    size=CFG['model']['hubert_size'],
    blocks_to_unfreeze=8,
)

auc_8b, hist_8b, _ = run_experiment(
    model_8b, train_ds, val_ds,
    experiment_name='hubert_ecg_blocks8',
    epochs=CFG['training']['epochs'],
    lr=CFG['training']['lr_pretrained'],
    batch_size=CFG['training']['batch_size_full'],
    save_dir=CFG['paths']['results'],
)
del model_8b; torch.cuda.empty_cache()

Loading HuBERT-ECG-base...
Loaded.
Unfrozen 128 tensors from last 8 blocks (layers 4–11)
Trainable: 56,902,661 / 93,323,397 (61.0%)

 Experiment : hubert_ecg_blocks8
 Device     : cuda
 Trainable  : 56,902,661  (61.0%)

Epoch 01/15  lr=5.00e-05
  train_loss=0.8176  val_loss=0.8218
  AUC (macro): 0.4603
  F1  (macro): 0.2132
  Per-class AUC:
    NORM : 0.446  ########
    MI   : 0.557  ###########
    STTC : 0.440  ########
    CD   : 0.341  ######
    HYP  : 0.517  ##########
Saved -> D:\GitHub\biosignal-xai\results/hubert_ecg_blocks8\best_adapter/checkpoint.pt
  * Best saved — AUC 0.4603

Epoch 02/15  lr=1.00e-04
  train_loss=0.5746  val_loss=0.6360
  AUC (macro): 0.8434
  F1  (macro): 0.5707
  Per-class AUC:
    NORM : 0.891  #################
    MI   : 0.878  #################
    STTC : 0.857  #################
    CD   : 0.845  ################
    HYP  : 0.745  ##############
Saved -> D:\GitHub\biosignal-xai\results/hubert_ecg_blocks8\best_adapter/checkpoint.pt
  * Best saved — 

## Experiment 2: LoRA r=8

Low-Rank Adaptation (LoRA) keeps the entire backbone **frozen** and injects pairs of small trainable matrices `A` (rank×d) and `B` (d×rank) into the attention Q and V projection layers. The effective weight update is `BA`, constrained to rank 8 — meaning only ~789 K parameters are trained regardless of backbone size.

A shape and parameter-count assertion is run before training to verify the PEFT configuration is correct (output shape `(B, 5)` and trainable fraction < 5%).

In [4]:
# Verify shape and parameter efficiency before committing to training
_probe = HuBERTECGPEFT(rank=8, use_dora=False).to(device)
_dummy = torch.randn(2, 12, 1000, device=device)
with torch.no_grad():
    _out = _probe(_dummy)
assert _out.shape == (2, 5), f'Shape error: {_out.shape}'
_p = _probe.count_parameters()
assert _p['trainable'] / _p['total'] < 0.05, \
    f"LoRA trainable fraction {_p['trainable']/_p['total']:.1%} exceeds expected <5%"
print(f'Shape OK: (2, 12, 1000) -> {_out.shape}')
print(f'Params:   {_p["trainable"]:,} / {_p["total"]:,} = {_p["percentage"]}')
del _probe, _dummy, _out
torch.cuda.empty_cache()

Loading HuBERT-ECG-base (LoRA r=8)...
Loaded.
HuBERTECGPEFT | Trainable: 789,509 / 93,913,221 (0.8%)
Shape OK: (2, 12, 1000) -> torch.Size([2, 5])
Params:   789,509 / 93,913,221 = 0.8%


In [5]:
model_lora = HuBERTECGPEFT(rank=8, use_dora=False)

auc_lora, hist_lora, _ = run_experiment(
    model_lora, train_ds, val_ds,
    experiment_name='hubert_ecg_lora_r8',
    epochs=CFG['training']['epochs'],
    lr=CFG['training']['lr_pretrained'],
    batch_size=CFG['training']['batch_size_full'],
    save_dir=CFG['paths']['results'],
)
del model_lora; torch.cuda.empty_cache()

Loading HuBERT-ECG-base (LoRA r=8)...
Loaded.
HuBERTECGPEFT | Trainable: 789,509 / 93,913,221 (0.8%)

 Experiment : hubert_ecg_lora_r8
 Device     : cuda
 Trainable  : 789,509  (0.8%)

Epoch 01/15  lr=5.00e-05
  train_loss=0.8250  val_loss=0.8289
  AUC (macro): 0.4776
  F1  (macro): 0.2712
  Per-class AUC:
    NORM : 0.619  ############
    MI   : 0.422  ########
    STTC : 0.520  ##########
    CD   : 0.339  ######
    HYP  : 0.488  #########
Saved -> D:\GitHub\biosignal-xai\results/hubert_ecg_lora_r8\best_adapter
  * Best saved — AUC 0.4776

Epoch 02/15  lr=1.00e-04
  train_loss=0.6314  val_loss=0.6968
  AUC (macro): 0.7996
  F1  (macro): 0.4499
  Per-class AUC:
    NORM : 0.870  #################
    MI   : 0.819  ################
    STTC : 0.823  ################
    CD   : 0.801  ################
    HYP  : 0.686  #############
Saved -> D:\GitHub\biosignal-xai\results/hubert_ecg_lora_r8\best_adapter
  * Best saved — AUC 0.7996

Epoch 03/15  lr=9.85e-05
  train_loss=0.5580  val_lo

## Experiment 3: DoRA r=8

Weight-Decomposed Low-Rank Adaptation (DoRA) extends LoRA by decomposing each adapted weight matrix into a **magnitude** scalar and a **direction** matrix. The direction is updated via LoRA while the magnitude is an independent trainable scalar per output feature. This decomposition more closely mirrors the update pattern of full fine-tuning, typically improving convergence stability.

DoRA adds ~37 K extra parameters over LoRA r=8 (one magnitude scalar per attention projection row) — still well under 1% of total backbone parameters.

In [6]:
model_dora = HuBERTECGPEFT(rank=8, use_dora=True)

auc_dora, hist_dora, _ = run_experiment(
    model_dora, train_ds, val_ds,
    experiment_name='hubert_ecg_dora_r8',
    epochs=CFG['training']['epochs'],
    lr=CFG['training']['lr_pretrained'],
    batch_size=CFG['training']['batch_size_full'],
    save_dir=CFG['paths']['results'],
)
del model_dora; torch.cuda.empty_cache()

print(f'\nHuBERT experiment summary:')
print(f'  Selective 8 blocks : AUC {auc_8b:.4f}')
print(f'  LoRA r=8           : AUC {auc_lora:.4f}')
print(f'  DoRA r=8           : AUC {auc_dora:.4f}')
print(f'\nFull comparison and plots -> 04c_comparison.ipynb')

Loading HuBERT-ECG-base (DoRA r=8)...
Loaded.
HuBERTECGPEFT | Trainable: 826,373 / 93,950,085 (0.9%)

 Experiment : hubert_ecg_dora_r8
 Device     : cuda
 Trainable  : 826,373  (0.9%)

Epoch 01/15  lr=5.00e-05
  train_loss=0.8301  val_loss=0.8169
  AUC (macro): 0.4873
  F1  (macro): 0.2726
  Per-class AUC:
    NORM : 0.558  ###########
    MI   : 0.463  #########
    STTC : 0.555  ###########
    CD   : 0.411  ########
    HYP  : 0.450  #########
Saved -> D:\GitHub\biosignal-xai\results/hubert_ecg_dora_r8\best_adapter
  * Best saved — AUC 0.4873

Epoch 02/15  lr=1.00e-04
  train_loss=0.6315  val_loss=0.6884
  AUC (macro): 0.7994
  F1  (macro): 0.4387
  Per-class AUC:
    NORM : 0.865  #################
    MI   : 0.816  ################
    STTC : 0.824  ################
    CD   : 0.808  ################
    HYP  : 0.685  #############
Saved -> D:\GitHub\biosignal-xai\results/hubert_ecg_dora_r8\best_adapter
  * Best saved — AUC 0.7994

Epoch 03/15  lr=9.85e-05
  train_loss=0.5548  val